# Making Plots of the data

Imports:

In [8]:
import sys
from pathlib import Path

root_dir = Path().resolve().parent
sys.path.append(str(root_dir))

from lib.types.report import BrainReport, StudyReport, StudyReportCollection
from models.download import Model
from lib import optimisation_helpers
from lib import utils

report_dir = root_dir / "reports"

def get_avg(file: Path) -> StudyReport:
    report = StudyReport.model_validate_json(file.read_text())
    return StudyReportCollection.average_study(report)

In [ ]:
d = {}
for i in range(7, 11):
    file = report_dir / f"v{i}_pet_llm_smollm2-1.7b-q8_0.json"
    report = get_avg(file)

    oob = report.simulation_config.brain.thoughts.out_of_bounds_message

    thought_loop_ratio = float("inf")
    best = None
    for trial in report.trials:
        config, result = trial.params, trial.report
        if result.iterations < 10:
            continue

        loss = result.thought_loops / result.iterations
        if loss < thought_loop_ratio:
            thought_loop_ratio = loss
            best = trial

            d[oob] = loss

    assert best is not None, "couldn't get trial"

    avg_list: list[BrainReport] = []
    for trial in report.trials:
        trial.params.seed = None
        if trial.params == best.params:
            avg_list.append(trial.report)

    print(f"best for {oob}: {sum([(x.iterations) for x in avg_list])}")

best for You can't leave the tank! Try a coordinate inside ({}, {}).: 24
best for You can't leave the tank! Ensure x coordinate is between 0 and {}, and y coordinate is between 0 and {}.: 28
best for You can't leave the tank!: 38
best for : 43


In [ ]:
d ={}
total_best_loss = float("inf")
total_best_model = ""

for model in [Model.smollm3, Model.smollm2, Model.llama, Model.granite, Model.deepseek]:
    report = get_avg(report_dir / f"v12_pet_llm_{model}.json")
    
    best_loss = float("inf")
    best = None
    for trial in report.trials:
        config, result = trial.params, trial.report
        if result.iterations < 10:
            continue

        loss = utils.loss_function(result, report.loss_function_weights)
        if loss < best_loss:
            best_loss = loss
            best = trial

            # d[model.name] = config
            d[model] = loss

        if loss < total_best_loss:
            total_best_loss = loss
            total_best_model = model.name

    assert best is not None, "couldn't get trial"


for k,v in d.items():
    print(k.name)
    print(f"    - {k.value}")
    print(f"    - loss: {v:.2f}")
print("\nbest is",total_best_model)



smollm3
    - SmolLM3-3B-128K-Q4_K_M
    - loss: 8.26
smollm2
    - smollm2-1.7b-q8_0
    - loss: 2.48
llama
    - llama-3.2-3b-q4_0
    - loss: 1.45
granite
    - granite-4.2-3b-Q4_K_M
    - loss: 5.90
deepseek
    - DeepSeek-R1-Distill-Qwen-1.5B-Q4_K_M
    - loss: 4.04

best is llama
